# 01 — Tokenizer Playground

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/marcoharuni/forge-tokenizer/blob/main/notebooks/tokenizer_playground.ipynb)

Train a tiny byte-level BPE tokenizer from scratch, inspect its tokens, and measure multilingual fertility.

**Runtime:** CPU is fine. No GPU required.

---

## 1. Install & Clone

In [ ]:
!pip install -q numpy matplotlib pytest tqdm regex
%cd /content
!test -d forge-tokenizer || git clone https://github.com/marcoharuni/forge-tokenizer.git
%cd /content/forge-tokenizer
!git pull --ff-only
!pip install -q -e .

## 2. Imports

In [ ]:
from pathlib import Path
import json

from forge_tokenizer import ForgeTokenizer
from forge_tokenizer.normalizer import explain_codepoints, normalize_text
from forge_tokenizer.pretokenizer import pretokenize_with_offsets
from forge_tokenizer.metrics import compression_ratio, fertility
from forge_tokenizer.visualization import plot_bpe_dropout_distribution, plot_fertility_bar

print('forge-tokenizer ready')

## 3. Unicode Normalization

Composed and decomposed text can look identical while using different code points.

In [ ]:
composed = 'é'
decomposed = 'e\u0301'

print(composed == decomposed)
print(normalize_text(composed) == normalize_text(decomposed))
explain_codepoints(decomposed)

## 4. Pretokenization

The simplified GPT-style regex keeps leading spaces attached where useful.

In [ ]:
text = ' hello, dunia! café 🔥'
for token, start, end in pretokenize_with_offsets(text):
    print(f'{start:02d}:{end:02d}', repr(token))

## 5. Train A Toy Byte-Level BPE Tokenizer

The tokenizer starts from UTF-8 bytes and learns deterministic merge rules from the local corpus.

In [ ]:
corpus = Path('data/tiny_corpus.txt').read_text(encoding='utf-8')
tokenizer = ForgeTokenizer().train(corpus, num_merges=80)

print(f'Vocabulary size: {tokenizer.vocab_size}')
print('First 10 merges:')
for left, right in tokenizer.merges[:10]:
    print(f'  {left!r} + {right!r} -> {left + right!r}')

## 6. Encode, Decode, Explain

In [ ]:
sample = 'Habari dunia, tokenizer sees café and 🔥.'
ids = tokenizer.encode(sample)
decoded = tokenizer.decode(ids)

print('Text:    ', sample)
print('IDs:     ', ids)
print('Decoded: ', decoded)
print('Roundtrip:', decoded == sample)

tokenizer.explain(sample)[:12]

## 7. Multilingual Fertility

These are local measurements from the tiny included corpus, not universal tokenizer claims.

In [ ]:
samples = json.loads(Path('data/multilingual_samples.json').read_text(encoding='utf-8'))
results = {}

for language, text in samples.items():
    results[language] = fertility(tokenizer, text)
    print(f'{language:12s} fertility={results[language]:6.3f} compression={compression_ratio(tokenizer, text):6.3f}')

plot_fertility_bar(results, 'generated/notebook_fertility.png')

## 8. BPE Dropout

In [ ]:
dropout_text = ' tokenization tokenization tokenization'
dropout_samples = [tokenizer.encode(dropout_text, dropout=0.2, seed=seed) for seed in range(40)]

print('First five samples:')
for ids in dropout_samples[:5]:
    print(ids)

print('Unique token counts:', sorted({len(ids) for ids in dropout_samples}))
plot_bpe_dropout_distribution(dropout_samples, 'generated/notebook_bpe_dropout.png')

## 9. Save And Reload

In [ ]:
output = Path('generated/notebook_tokenizer.json')
output.parent.mkdir(exist_ok=True)
tokenizer.save(output)

loaded = ForgeTokenizer.load(output)
print('Saved to:', output)
print('Same encoding:', loaded.encode(sample) == tokenizer.encode(sample))

---

**Next →** [Embedding Geometry](embedding_geometry.ipynb)